In [3]:
from cellphonedb.src.core.methods import cpdb_analysis_method
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
from cellphonedb.src.core.methods import cpdb_degs_analysis_method
import cellphonedb
import fastccc
import scanpy as sc
import pandas as pd
import ktplotspy as kpy
import matplotlib.pyplot as plt
from glob import glob
from joblib import Parallel, delayed

# 所有50 M细胞分年龄段

In [3]:
adata = sc.read_h5ad('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_scRNA_count.h5ad',backed=True)

In [4]:
indices = pd.read_csv("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_indices_labels_umap.csv",index_col=0)

In [5]:
if (adata.obs.index == indices.index).all():
    adata.obs = adata.obs.join(indices,how='left')
else:
    raise ValueError('Indices error.')

In [6]:
indices = adata.obs.copy()

In [7]:
adata.strings_to_categoricals()

In [8]:
adata

AnnData object with n_obs × n_vars = 51898839 × 38606 backed at '/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_scRNA_count.h5ad'
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Reference_Atlas_L1L2_mv', 'Reference_Atlas_L1L2_pl', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'percent.mt', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'mapping.score', 'scDblFinder.class', 'dataset', 'Classification_L4', 'Classification_L3', 'Classification_L2', 'Classification_L1', 'leiden_cluster', 'UMAP_1', 'UMAP_2'

In [9]:
Level_counts_frac =  pd.read_excel("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_sample_counts_frac_clinicl_table.xlsx",
                                   index_col=0)
info_cols = [
        'SampleID',"ClinicalID",'DonorID', "Gender", "Ten_year_intervals",
        "Pregnancy", 'Pregnancy stage', "Sampling time",'Age_in_Years'
    ]
metadata = Level_counts_frac[info_cols]

In [10]:
mapping_dict = dict(zip(metadata['SampleID'], metadata['Ten_year_intervals']))
indices['Ten_year_intervals'] = indices['SampleID'].map(mapping_dict)

In [11]:
metadata_valid = metadata[(metadata["Pregnancy"] == "NO") & (metadata["Sampling time"] == "Morning")].copy()

## 过滤不需要的人

In [12]:
indices = indices[indices["SampleID"].isin(metadata_valid.SampleID)]

## 过滤所有年龄段都>200 cells的细胞类型

In [13]:
cell_counts = indices.groupby(['Classification_L4', 'Ten_year_intervals']).size().unstack(fill_value=0)

In [14]:
cell_counts

Ten_year_intervals,10-19,2-9,20-29,30-39,40-49,50-59,60-69,70-79,80-89,90+
Classification_L4,,,,,,,,,,
ALPL- MARCKS- NDNs,124144,84950,110573,191218,157483,105802,290056,369300,432744,239364
ASDC,337,377,413,569,539,286,596,672,734,365
Adaptive NK cells,82756,82208,112597,213992,183791,147165,291626,393294,499630,264914
Atypical naïve B cells,628,1005,477,1115,1036,1034,1618,1792,1618,756
Basophils,2190,2266,3278,5067,3518,2250,3541,3293,4357,1696
...,...,...,...,...,...,...,...,...,...,...
VIM- FLNA- NDNs,127374,85999,154755,261658,228679,142421,260404,343430,323393,164371
cDC1,559,459,559,924,916,486,965,999,1064,442
iNKT,61,53,59,81,84,50,151,83,93,28


In [15]:
valid_cell_types = cell_counts[(cell_counts > 200).all(axis=1)].index.tolist()
len(valid_cell_types)

73

In [16]:
stages = list(metadata_valid["Ten_year_intervals"].unique())

In [ ]:
for stage in stages:
    stage_indices = indices[(indices["Ten_year_intervals"] == stage) & (indices["Classification_L4"].isin(valid_cell_types))]
    print(stage_indices.shape)
    #stage_indices.to_csv(f"/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/{stage}_indices.csv")
    stage_adata = adata[stage_indices.index]
    stage_adata.write(f"/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/{stage}_intervals.h5ad",compression='gzip')

In [ ]:
# restart kernel

# 全部基因放入cellphonedb或FastCCC分析

In [2]:
def process_file(ad_path):
    adata = sc.read_h5ad(ad_path)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.write(ad_path.split('.')[0] + "_normalised_log.h5ad",
                compression='gzip')

In [3]:
files = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/*.h5ad')
Parallel(n_jobs=10)(
    delayed(process_file)(f) for f in files
)

[None, None, None, None, None, None, None, None, None, None]

## fastccc 单一CS检测

In [ ]:
# single_unit_summary: median最严格，显著的对儿最少，其次是Q3, Quantile_0.9, mean最松弛

In [22]:
LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
save_path = '/home/liyanguo/MyImmuCell/12_FastCCC/results1'

In [ ]:
interactions_strength, pvals, percents_analysis = fastccc.statistical_analysis_method(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = '/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/2-9_intervals_normalised_log.h5ad',
    convert_type = 'hgnc_symbol',
    single_unit_summary = 'Q3',
    complex_aggregation = 'Minimum',
    LR_combination = 'Arithmetic',
    min_percentile = 0.1,
    meta_key = 'Classification_L4',
    use_DEG = True,
    save_path = save_path
)

## fastccc 10个年龄段、16种CS检测 L4

In [34]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L4/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.Cauchy_combination_of_statistical_analysis_methods(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary_list = ['Mean', 'Median', 'Q3', 'Quantile_0.9'],
    complex_aggregation_list = ['Minimum', 'Average'],
    LR_combination_list = ['Arithmetic', 'Geometric'],
    min_percentile = 0.1,
    meta_key = 'Classification_L4',
    use_DEG = False,
    save_path = save_path)

In [35]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-02-27 16:24:11 | INFO     | Task id is ed853a.
2026-02-27 16:24:11 | INFO     | Task id is 02679f.
2026-02-27 16:24:11 | INFO     | Task id is 7a4769.
2026-02-27 16:24:11 | INFO     | Task id is e4b534.
2026-02-27 16:24:11 | INFO     | Task id is b148ae.
2026-02-27 16:24:11 | INFO     | Task id is 79f350.
2026-02-27 16:24:11 | INFO     | Task id is dcb0c3.
2026-02-27 16:24:11 | INFO     | Task id is 746063.
2026-02-27 16:24:11 | INFO     | Directory already exists: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS/20-29
2026-02-27 16:24:11 | INFO     | Directory already exists: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS/60-69
2026-02-27 16:24:11 | INFO     | Directory already exists: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS/50-59
2026-02-27 16:24:11 | INFO     | Directory already exists: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS/80-89
2026-02-27 16:24:11 | INFO     | Directory already exists: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS/90+
2026-0

[None, None, None, None, None, None, None, None, None, None]

## fastccc 10个年龄段、16种CS检测 L3

In [36]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.Cauchy_combination_of_statistical_analysis_methods(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary_list = ['Mean', 'Median', 'Q3', 'Quantile_0.9'],
    complex_aggregation_list = ['Minimum', 'Average'],
    LR_combination_list = ['Arithmetic', 'Geometric'],
    min_percentile = 0.1,
    meta_key = 'Classification_L3',
    use_DEG = False,
    save_path = save_path)

In [37]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-02-28 09:04:30 | INFO     | Task id is 7d7312.
2026-02-28 09:04:30 | INFO     | Task id is e79e15.
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/10-19
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/70-79
2026-02-28 09:04:30 | INFO     | Task id is 16d393.
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/90+
2026-02-28 09:04:30 | INFO     | Task id is 0861c7.
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/60-69
2026-02-28 09:04:30 | INFO     | Task id is 65996d.
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/50-59
2026-02-28 09:04:30 | INFO     | Task id is ff210b.
2026-02-28 09:04:30 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_16_CS_L3/30-39
2026-02-28 09:04:3

[None, None, None, None, None, None, None, None, None, None]

## fastccc 10个年龄段、同cellphonedb 检测 L4.
Mean的结果不好，一个例子是FPR1是髓系特异性的，Mean识别有误。median可以正确识别

In [50]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L4/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.statistical_analysis_method(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary  = 'Median',
    complex_aggregation = 'Minimum',
    LR_combination = 'Arithmetic',
    min_percentile = 0.1,
    meta_key = 'Classification_L4',
    use_DEG = False,
    save_path = save_path)

In [51]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-03-09 16:45:50 | INFO     | Task id is ed6391.
2026-03-09 16:45:50 | INFO     | Task id is 83b58d.
2026-03-09 16:45:50 | INFO     | Task id is 031d40.
2026-03-09 16:45:50 | INFO     | Task id is 97a7ce.
2026-03-09 16:45:50 | INFO     | Task id is 3bcbc6.
2026-03-09 16:45:50 | INFO     | Task id is 40add1.
2026-03-09 16:45:50 | INFO     | Task id is 3a4e02.
2026-03-09 16:45:50 | INFO     | Task id is 00cd28.
2026-03-09 16:45:50 | INFO     | Task id is b4ef38.
2026-03-09 16:45:50 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L4/10-19
2026-03-09 16:45:50 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L4/30-39
2026-03-09 16:45:50 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L4/50-59
2026-03-09 16:45:50 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L4/60-69
2026-03-09 16:45:50 | INFO     | Task id is 273e5c.
2026-03-

[None, None, None, None, None, None, None, None, None, None]

## fastccc 10个年龄段、同cellphonedb 检测 L3

In [52]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.statistical_analysis_method(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary  = 'Median',
    complex_aggregation = 'Minimum',
    LR_combination = 'Arithmetic',
    min_percentile = 0.1,
    meta_key = 'Classification_L3',
    use_DEG = False,
    save_path = save_path)

In [53]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-03-09 17:05:01 | INFO     | Task id is 4ac901.
2026-03-09 17:05:01 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/90+
2026-03-09 17:05:01 | INFO     | Task id is 13adcd.
2026-03-09 17:05:01 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/10-19
2026-03-09 17:05:01 | INFO     | Task id is 4d857e.
2026-03-09 17:05:01 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/70-79
2026-03-09 17:05:04 | INFO     | Task id is 3f31a2.
2026-03-09 17:05:04 | INFO     | Task id is 227021.
2026-03-09 17:05:04 | INFO     | Task id is b9ea42.
2026-03-09 17:05:04 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/20-29
2026-03-09 17:05:04 | INFO     | Task id is dd7957.
2026-03-09 17:05:04 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC/results_median_CPDB_L3/80-89
2026-03-09 17:05:04 | INFO     | Directory creat

[None, None, None, None, None, None, None, None, None, None]

# 所有怀孕队列

In [4]:
adata = sc.read_h5ad('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_scRNA_count.h5ad',backed=True)

In [5]:
indices = pd.read_csv("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_indices_labels_umap.csv",index_col=0)

In [6]:
if (adata.obs.index == indices.index).all():
    adata.obs = adata.obs.join(indices,how='left')
else:
    raise ValueError('Indices error.')

In [7]:
indices = adata.obs.copy()

In [8]:
adata.strings_to_categoricals()

In [9]:
adata

AnnData object with n_obs × n_vars = 51898839 × 38606 backed at '/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Full_dataset/Full_dataset_scRNA_count.h5ad'
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Reference_Atlas_L1L2_mv', 'Reference_Atlas_L1L2_pl', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'percent.mt', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'mapping.score', 'scDblFinder.class', 'dataset', 'Classification_L4', 'Classification_L3', 'Classification_L2', 'Classification_L1', 'leiden_cluster', 'UMAP_1', 'UMAP_2'

In [10]:
Level_counts_frac =  pd.read_excel("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_sample_counts_frac_clinicl_table.xlsx",
                                   index_col=0)
info_cols = [
        'SampleID',"ClinicalID",'DonorID', "Gender", "Ten_year_intervals",
        "Pregnancy", 'Pregnancy stage', "Sampling time",'Age_in_Years'
    ]
metadata = Level_counts_frac[info_cols]

In [29]:
pregnant = metadata[(metadata['Pregnancy'] == 'YES') & (metadata['Pregnancy stage'] != 'Parturition')]

In [30]:
non_pregnant = metadata[
        (metadata['Pregnancy'] == 'NO') & 
        (metadata['Age_in_Years'] >= 18) & 
        (metadata['Age_in_Years'] <= 40) & 
        (metadata['Gender'] == 'Female')
    ]

In [31]:
mapping_dict = dict(zip(metadata['SampleID'], metadata['Pregnancy stage']))
indices['Pregnancy stage'] = indices['SampleID'].map(mapping_dict)

/tmp/ipykernel_2900326/2350218894.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [32]:
metadata_valid = metadata[(metadata['SampleID'].isin(pregnant.SampleID)) | (metadata['SampleID'].isin(non_pregnant.SampleID))].copy()

## 过滤不需要的人

In [33]:
indices = indices[indices["SampleID"].isin(metadata_valid.SampleID)]

In [34]:
indices['Pregnancy stage'].value_counts()

Pregnancy stage
Unavailable         5042189
First trimester     1029430
Second trimester     677220
Third trimester      323347
Name: count, dtype: int64

## 过滤所有年龄段都>200 cells的细胞类型

In [35]:
cell_counts = indices.groupby(['Classification_L4', 'Pregnancy stage']).size().unstack(fill_value=0)

In [36]:
cell_counts

Pregnancy stage,First trimester,Second trimester,Third trimester,Unavailable
Classification_L4,,,,
ALPL- MARCKS- NDNs,33139,19328,8726,202331
ASDC,74,53,28,477
Adaptive NK cells,19712,13585,8645,185013
Atypical naïve B cells,122,62,45,784
Basophils,610,713,249,4141
...,...,...,...,...
VIM- FLNA- NDNs,62327,28865,11494,256789
cDC1,47,60,35,653
iNKT,9,8,6,96


In [37]:
valid_cell_types = cell_counts[(cell_counts > 200).all(axis=1)].index.tolist()
len(valid_cell_types)

53

In [39]:
stages = list(metadata_valid["Pregnancy stage"].unique())

In [40]:
for stage in stages:
    stage_indices = indices[(indices["Pregnancy stage"] == stage) & (indices["Classification_L4"].isin(valid_cell_types))]
    print(stage_indices.shape)
    #stage_indices.to_csv(f"/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Age_intervals/{stage}_indices.csv")
    stage_adata = adata[stage_indices.index]
    stage_adata.write(f"/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Pregnancy_stage/{stage}.h5ad",compression='gzip')

(5003821, 44)
(1023683, 44)
(669453, 44)
(319192, 44)


# 全部基因放入cellphonedb或FastCCC分析

In [41]:
def process_file(ad_path):
    adata = sc.read_h5ad(ad_path)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.write(ad_path.split('.')[0] + "_normalised_log.h5ad",
                compression='gzip')

In [42]:
files = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Pregnancy_stage/*.h5ad')
Parallel(n_jobs=10)(
    delayed(process_file)(f) for f in files
)

[None, None, None, None]

## fastccc 10个年龄段、16种CS检测 L4

In [44]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L4/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.Cauchy_combination_of_statistical_analysis_methods(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary_list = ['Mean', 'Median', 'Q3', 'Quantile_0.9'],
    complex_aggregation_list = ['Minimum', 'Average'],
    LR_combination_list = ['Arithmetic', 'Geometric'],
    min_percentile = 0.1,
    meta_key = 'Classification_L4',
    use_DEG = False,
    save_path = save_path)

In [45]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Pregnancy_stage/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-03-21 11:13:59 | INFO     | Task id is fc213e.
2026-03-21 11:13:59 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L4/Unavailable
2026-03-21 11:14:00 | INFO     | Task id is 16a395.
2026-03-21 11:14:00 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L4/Third trimester
2026-03-21 11:14:00 | INFO     | Task id is 2c82be.
2026-03-21 11:14:00 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L4/First trimester
2026-03-21 11:14:00 | INFO     | Task id is 43b36f.
2026-03-21 11:14:00 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L4/Second trimester
2026-03-21 11:14:18 | SUCCESS  | Data preprocessing done.
2026-03-21 11:14:34 | INFO     | Running:
-> Mean for single-unit summary function.
-> Minimum for multi-unit complex aggregation.
-> Arithmetic for L-R combination to compute the CS.
-> Percentile is 0.1.
2026-03-21 11:14:39 | SUCCESS  | Da

[None, None, None, None]

## fastccc 10个年龄段、16种CS检测 L3

In [46]:
def FastCCC_job(counts_file_path):
    LRI_db_file_path = '/home/liyanguo/software/FastCCC-0.1.2/db/CPDBv5.0.0'
    save_path = '/home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L3/' + counts_file_path.split('/')[-1].split('_')[0]

    fastccc.Cauchy_combination_of_statistical_analysis_methods(
    database_file_path = LRI_db_file_path,
    celltype_file_path = None,
    counts_file_path = counts_file_path,
    convert_type = 'hgnc_symbol',
    single_unit_summary_list = ['Mean', 'Median', 'Q3', 'Quantile_0.9'],
    complex_aggregation_list = ['Minimum', 'Average'],
    LR_combination_list = ['Arithmetic', 'Geometric'],
    min_percentile = 0.1,
    meta_key = 'Classification_L3',
    use_DEG = False,
    save_path = save_path)

In [47]:
counts_file_paths = glob('/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Pregnancy_stage/*_normalised_log.h5ad')
Parallel(n_jobs=10)(
    delayed(FastCCC_job)(f) for f in counts_file_paths
)

2026-03-21 11:36:15 | INFO     | Task id is ffefa4.
2026-03-21 11:36:15 | INFO     | Task id is 7f4e8e.
2026-03-21 11:36:15 | INFO     | Task id is af07d2.
2026-03-21 11:36:15 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L3/First trimester
2026-03-21 11:36:15 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L3/Second trimester
2026-03-21 11:36:15 | INFO     | Task id is 47ec8f.
2026-03-21 11:36:15 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L3/Unavailable
2026-03-21 11:36:15 | INFO     | Directory created: /home/liyanguo/MyImmuCell/12_FastCCC_P/results_16_CS_L3/Third trimester
2026-03-21 11:36:32 | SUCCESS  | Data preprocessing done.
2026-03-21 11:36:47 | INFO     | Running:
-> Mean for single-unit summary function.
-> Minimum for multi-unit complex aggregation.
-> Arithmetic for L-R combination to compute the CS.
-> Percentile is 0.1.
2026-03-21 11:36:49 | SUCCESS  | CS

[None, None, None, None]